In [0]:
%pip install databricks-sdk --upgrade --quiet
dbutils.library.restartPython()

# Step 2: Create & Tune Two Genie Spaces

This notebook creates **two** Genie spaces for the supply chain supervisor agent:

| Space | Domain | Tables | Metric View |
|-------|--------|--------|---------|
| **Procurement & Inventory** | Inbound materials | suppliers, materials, purchase_orders, inventory | procurement_metrics |
| **Logistics & Fulfillment** | Outbound shipments | carriers, shipments, routes, delivery_events | logistics_metrics |

**Tuning approach:**
- Metric views handle joins, measures, and dimensions → no duplicate knowledge snippets
- Text instructions cover business conventions the metric view can't express
- Sample SQL queries show verified patterns for common questions
- Filters cover edge-case row selection not in the metric view

**Prerequisites:** Run `01_create_supply_chain_data` first.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import CreateWarehouseRequestWarehouseType
import json
import uuid

w = WorkspaceClient()

# ============================================================
# SINGLE CONFIG BLOCK - only CATALOG is hardcoded
# ============================================================
CATALOG = "agents"
SCHEMA = "supply_chain_demo"
WAREHOUSE_NAME = "supply-chain-demo-wh"

# Dynamically find a SQL warehouse, or create a serverless one
warehouses = list(w.warehouses.list())
if warehouses:
    # Prefer a warehouse matching our name, then any shared, then first available
    named = [wh for wh in warehouses if wh.name == WAREHOUSE_NAME]
    shared = [wh for wh in warehouses if "shared" in (wh.name or "").lower()]
    wh = (named[0] if named else shared[0] if shared else warehouses[0])
    WAREHOUSE_ID = wh.id
    print(f"Using existing warehouse: {wh.name} ({WAREHOUSE_ID})")
else:
    # Create the smallest possible serverless SQL warehouse
    print(f"No warehouses found. Creating serverless warehouse: {WAREHOUSE_NAME}")
    created = w.warehouses.create_and_wait(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        warehouse_type=CreateWarehouseRequestWarehouseType.PRO,
        enable_serverless_compute=True,
        auto_stop_mins=5,
        min_num_clusters=1,
        max_num_clusters=1,
    )
    WAREHOUSE_ID = created.id
    print(f"✓ Created serverless warehouse: {WAREHOUSE_NAME} ({WAREHOUSE_ID})")

# Space 1: Procurement & Inventory
SPACE_1_TITLE = "Supply Chain - Procurement & Inventory"
SPACE_1_TABLES = sorted([
    f"{CATALOG}.{SCHEMA}.suppliers",
    f"{CATALOG}.{SCHEMA}.materials",
    f"{CATALOG}.{SCHEMA}.purchase_orders",
    f"{CATALOG}.{SCHEMA}.inventory",
    f"{CATALOG}.{SCHEMA}.procurement_metrics",
])

# Space 2: Logistics & Fulfillment
SPACE_2_TITLE = "Supply Chain - Logistics & Fulfillment"
SPACE_2_TABLES = sorted([
    f"{CATALOG}.{SCHEMA}.carriers",
    f"{CATALOG}.{SCHEMA}.shipments",
    f"{CATALOG}.{SCHEMA}.routes",
    f"{CATALOG}.{SCHEMA}.delivery_events",
    f"{CATALOG}.{SCHEMA}.logistics_metrics",
])

print(f"\nSpace 1: {SPACE_1_TITLE}")
for t in SPACE_1_TABLES:
    print(f"  - {t}")
print(f"\nSpace 2: {SPACE_2_TITLE}")
for t in SPACE_2_TABLES:
    print(f"  - {t}")

---
## Create Genie Spaces
Uses the REST API with `serialized_space` (version 2 format, tables sorted by identifier).

In [0]:
def find_or_create_space(title, tables, description):
    """Find existing space by title, or create a new one. Updates warehouse on existing spaces."""
    response = w.genie.list_spaces()
    existing = response.spaces or []
    matching = [s for s in existing if s.title == title]
    
    if matching:
        space_id = matching[0].space_id
        # Update warehouse to current one (handles deleted/changed warehouse)
        w.genie.update_space(
            space_id=space_id,
            title=title,
            warehouse_id=WAREHOUSE_ID,
        )
        print(f"ℹ️  Already exists: {title} ({space_id}) — warehouse updated to {WAREHOUSE_ID}")
        return space_id
    
    serialized_space = json.dumps({
        "version": 2,
        "data_sources": {
            "tables": [{"identifier": t} for t in tables]
        },
        "instructions": {}
    })
    
    result = w.api_client.do(
        method="POST",
        path="/api/2.0/genie/spaces",
        body={
            "title": title,
            "description": description,
            "warehouse_id": WAREHOUSE_ID,
            "serialized_space": serialized_space,
        }
    )
    space_id = result["space_id"]
    print(f"✓ Created: {title} ({space_id})")
    return space_id

# Create Space 1
SPACE_1_DESC = """Answers questions about procurement: suppliers, materials, purchase orders, and warehouse inventory.
Use the procurement_metrics view for spend, lead times, and on-time delivery.
Query inventory table directly for current stock levels and reorder alerts."""

SPACE_1_ID = find_or_create_space(SPACE_1_TITLE, SPACE_1_TABLES, SPACE_1_DESC)

# Create Space 2
SPACE_2_DESC = """Answers questions about logistics: carriers, shipments, routes, and delivery tracking.
Use the logistics_metrics view for shipping costs, transit times, and on-time rates.
Query delivery_events for real-time shipment tracking and delay details."""

SPACE_2_ID = find_or_create_space(SPACE_2_TITLE, SPACE_2_TABLES, SPACE_2_DESC)

print(f"\nSpace IDs:")
print(f"  Procurement: {SPACE_1_ID}")
print(f"  Logistics:   {SPACE_2_ID}")

---
## Tune Both Spaces
Add text instructions, sample SQL queries, and filters. Only add knowledge that the metric view doesn't already capture.

In [0]:
def update_space_config(space_id, instructions_text, sample_queries, filters):
    """Update a space's serialized_space with instructions, SQL examples, and filters."""
    # Get current config
    current = w.api_client.do(
        method="GET",
        path=f"/api/2.0/genie/spaces/{space_id}",
        query={"include_serialized_space": "true"}
    )
    config = json.loads(current["serialized_space"])
    config.setdefault("instructions", {})
    
    # Text instructions
    config["instructions"]["text_instructions"] = [{
        "id": uuid.uuid4().hex,
        "content": [instructions_text]
    }]
    
    # Sample SQL queries (sorted by id)
    example_sqls = [
        {"id": uuid.uuid4().hex, "question": [q["question"]], "sql": [q["sql"]]}
        for q in sample_queries
    ]
    config["instructions"]["example_question_sqls"] = sorted(example_sqls, key=lambda x: x["id"])
    
    # Filters (sorted by id)
    if filters:
        config["instructions"].setdefault("sql_snippets", {})
        filter_list = [
            {"id": uuid.uuid4().hex, "sql": [f["sql"]], "display_name": f["display_name"], "synonyms": f.get("synonyms", [])}
            for f in filters
        ]
        config["instructions"]["sql_snippets"]["filters"] = sorted(filter_list, key=lambda x: x["id"])
    
    # Push update
    w.api_client.do(
        method="PATCH",
        path=f"/api/2.0/genie/spaces/{space_id}",
        body={"serialized_space": json.dumps(config)}
    )

# ============================================================
# SPACE 1: Procurement & Inventory
# ============================================================

procurement_instructions = """- All costs are in USD.
- Use procurement_metrics view for spend, lead time, and on-time delivery questions — it handles joins and excludes cancelled POs.
- "Spend" or "total cost" means SUM(quantity * unit_cost) from purchase orders.
- Supplier reliability_score is 0-100 (higher = better). >90 is considered reliable.
- Lead time is measured in calendar days from order placement to delivery.
- On-time means actual_delivery <= expected_delivery.
- Warehouses are WH-EAST, WH-WEST, WH-CENTRAL.
- A material is "below reorder point" when quantity_on_hand < reorder_point.
- A material is "critically low" when quantity_on_hand < safety_stock.
- For raw inventory queries, use the inventory table directly (not the metric view)."""

procurement_queries = [
    {
        "question": "What is the total procurement spend by supplier?",
        "sql": f"SELECT `supplier_name`, MEASURE(`Total Spend`) AS spend, MEASURE(`Order Count`) AS orders FROM {CATALOG}.{SCHEMA}.procurement_metrics GROUP BY ALL ORDER BY spend DESC"
    },
    {
        "question": "Which suppliers have the worst on-time delivery?",
        "sql": f"SELECT `supplier_name`, `country`, MEASURE(`On-Time Delivery Rate`) AS otd_pct, MEASURE(`Order Count`) AS orders FROM {CATALOG}.{SCHEMA}.procurement_metrics GROUP BY ALL ORDER BY otd_pct ASC"
    },
    {
        "question": "What is the average lead time by country?",
        "sql": f"SELECT `country`, MEASURE(`Avg Lead Time`) AS avg_lead_days, MEASURE(`Order Count`) AS orders FROM {CATALOG}.{SCHEMA}.procurement_metrics GROUP BY ALL ORDER BY avg_lead_days DESC"
    },
    {
        "question": "Which materials are below reorder point?",
        "sql": f"SELECT i.warehouse_id, m.material_name, m.category, i.quantity_on_hand, i.reorder_point, i.safety_stock FROM {CATALOG}.{SCHEMA}.inventory i JOIN {CATALOG}.{SCHEMA}.materials m ON i.material_id = m.material_id WHERE i.quantity_on_hand < i.reorder_point ORDER BY (i.reorder_point - i.quantity_on_hand) DESC"
    },
    {
        "question": "What is the monthly spend trend?",
        "sql": f"SELECT `order_month`, MEASURE(`Total Spend`) AS spend, MEASURE(`Order Count`) AS orders FROM {CATALOG}.{SCHEMA}.procurement_metrics GROUP BY ALL ORDER BY `order_month`"
    },
]

procurement_filters = [
    {"sql": "purchase_orders.status = 'In Transit'", "display_name": "in-transit orders", "synonyms": ["open orders", "pending delivery"]},
    {"sql": "inventory.quantity_on_hand < inventory.reorder_point", "display_name": "below reorder point", "synonyms": ["needs reorder", "low stock"]},
    {"sql": "inventory.quantity_on_hand < inventory.safety_stock", "display_name": "critically low stock", "synonyms": ["stockout risk", "critical inventory"]},
]

update_space_config(SPACE_1_ID, procurement_instructions, procurement_queries, procurement_filters)
print(f"✓ Space 1 tuned: {SPACE_1_TITLE}")
print(f"  - {len(procurement_instructions.strip().splitlines())} text instructions")
print(f"  - {len(procurement_queries)} sample SQL queries")
print(f"  - {len(procurement_filters)} knowledge filters")

In [0]:
# ============================================================
# SPACE 2: Logistics & Fulfillment
# ============================================================

logistics_instructions = """- All costs are in USD. Weights are in kilograms.
- Use logistics_metrics view for shipping cost, transit time, and on-time rate questions — it handles joins and excludes returned shipments.
- "Shipping cost" or "freight cost" means the total shipping_cost from shipments.
- Transport modes: Truck (domestic, fast), Rail (domestic, cheap), Air (fastest, expensive), Ocean (international, slow/cheap).
- On-time means actual_delivery <= estimated_delivery.
- Warehouses: WH-EAST (Northeast US), WH-WEST (West Coast), WH-CENTRAL (Midwest).
- For shipment tracking/status updates, query delivery_events directly.
- A shipment is "delayed" when status = 'Delayed' or actual_delivery > estimated_delivery.
- Service regions: Domestic (US only), International (overseas), Both.
- For route planning questions, use the routes table."""

logistics_queries = [
    {
        "question": "What is the total shipping cost by carrier?",
        "sql": f"SELECT `carrier_name`, `transport_mode`, MEASURE(`Total Shipping Cost`) AS cost, MEASURE(`Shipment Count`) AS shipments FROM {CATALOG}.{SCHEMA}.logistics_metrics GROUP BY ALL ORDER BY cost DESC"
    },
    {
        "question": "What is the on-time delivery rate by transport mode?",
        "sql": f"SELECT `transport_mode`, MEASURE(`On-Time Rate`) AS otd_pct, MEASURE(`Shipment Count`) AS shipments, MEASURE(`Avg Transit Days`) AS avg_days FROM {CATALOG}.{SCHEMA}.logistics_metrics GROUP BY ALL ORDER BY otd_pct DESC"
    },
    {
        "question": "Which shipments are currently delayed?",
        "sql": f"SELECT sh.shipment_id, c.carrier_name, sh.origin_warehouse, sh.destination_city, sh.ship_date, sh.estimated_delivery, sh.status FROM {CATALOG}.{SCHEMA}.shipments sh JOIN {CATALOG}.{SCHEMA}.carriers c ON sh.carrier_id = c.carrier_id WHERE sh.status = 'Delayed' ORDER BY sh.estimated_delivery"
    },
    {
        "question": "What is the average cost per kilogram by carrier?",
        "sql": f"SELECT `carrier_name`, `transport_mode`, MEASURE(`Avg Cost per KG`) AS cost_per_kg, MEASURE(`Total Weight`) AS total_kg FROM {CATALOG}.{SCHEMA}.logistics_metrics GROUP BY ALL ORDER BY cost_per_kg DESC"
    },
    {
        "question": "Show the latest tracking events for a shipment",
        "sql": f"SELECT de.event_timestamp, de.event_type, de.location, de.notes FROM {CATALOG}.{SCHEMA}.delivery_events de WHERE de.shipment_id = 'SH001' ORDER BY de.event_timestamp DESC"
    },
    {
        "question": "What are the longest routes by distance?",
        "sql": f"SELECT route_id, origin, destination, distance_km, avg_transit_days, transport_mode FROM {CATALOG}.{SCHEMA}.routes ORDER BY distance_km DESC"
    },
]

logistics_filters = [
    {"sql": "shipments.status = 'Delayed'", "display_name": "delayed shipments", "synonyms": ["late shipments", "overdue"]},
    {"sql": "shipments.status = 'In Transit'", "display_name": "in-transit shipments", "synonyms": ["active shipments", "en route"]},
    {"sql": "carriers.transport_mode = 'Air'", "display_name": "air freight only", "synonyms": ["air shipments", "air cargo"]},
    {"sql": "delivery_events.event_type = 'Delay Reported'", "display_name": "delay events", "synonyms": ["delays", "delay reports"]},
]

update_space_config(SPACE_2_ID, logistics_instructions, logistics_queries, logistics_filters)
print(f"✓ Space 2 tuned: {SPACE_2_TITLE}")
print(f"  - {len(logistics_instructions.strip().splitlines())} text instructions")
print(f"  - {len(logistics_queries)} sample SQL queries")
print(f"  - {len(logistics_filters)} knowledge filters")

---
## Validate Both Spaces

In [0]:
print("=" * 60)
print("GENIE SPACE VALIDATION")
print("=" * 60)

for space_id, title, expected_tables in [
    (SPACE_1_ID, SPACE_1_TITLE, SPACE_1_TABLES),
    (SPACE_2_ID, SPACE_2_TITLE, SPACE_2_TABLES),
]:
    result = w.api_client.do(
        method="GET",
        path=f"/api/2.0/genie/spaces/{space_id}",
        query={"include_serialized_space": "true"}
    )
    config = json.loads(result["serialized_space"])
    tables = [t["identifier"] for t in config.get("data_sources", {}).get("tables", [])]
    instructions = config.get("instructions", {})
    
    print(f"\n┌ {'=' * 56} ┐")
    print(f"│  {title:<54} │")
    print(f"│  ID: {space_id:<49} │")
    print(f"├ {'=' * 56} ┤")
    
    # Check tables
    missing = set(expected_tables) - set(tables)
    print(f"│  Tables: {len(tables)}/{ len(expected_tables)} attached{'  ✓' if not missing else '  ✗ MISSING: ' + str(missing):<30} │")
    for t in tables:
        is_mv = "metrics" in t
        print(f"│    {'*' if is_mv else '-'} {t:<50} │")
    
    # Check instructions
    n_text = len(instructions.get("text_instructions", []))
    n_sql = len(instructions.get("example_question_sqls", []))
    n_filters = len(instructions.get("sql_snippets", {}).get("filters", []))
    print(f"│  Instructions: {n_text} text, {n_sql} SQL examples, {n_filters} filters  │")
    print(f"└ {'=' * 56} ┘")

print("\n✓ Both spaces ready! Use these IDs in notebook 03 (supervisor agent):")
print(f"  PROCUREMENT_SPACE_ID = \"{SPACE_1_ID}\"")
print(f"  LOGISTICS_SPACE_ID  = \"{SPACE_2_ID}\"")

---
## Next Steps

Both Genie spaces are created and tuned. Run **`03_supervisor_agent`** to build the agent that routes questions between them.

The supervisor will decide:
- *"Which supplier has late deliveries?"* → Procurement space
- *"What's our cheapest carrier?"* → Logistics space
- *"Are we at risk of stockout?"* → Procurement space
- *"Where is shipment SH017?"* → Logistics space